# Phase 13a: Model Evaluation — Hyperparameter Tuning

GridSearchCV tuning of the best-performing model (XGBoost) over n_estimators, max_depth, and learning_rate.

## Setup

In [1]:
   %pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd, numpy as np, joblib, json, time
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, roc_curve)

t0 = time.time()
X_train, X_test, y_train, y_test = joblib.load('../data/train_test_split.joblib')

numeric_features = ['qty', 'mass_kg', 'year', 'month', 'day', 'day_of_week', 'quarter', 'is_weekend']
categorical_features = ['debtor_code', 'product_code_grp', 'doc_type']

numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num', numeric_transformer, numeric_features), ('cat', categorical_transformer, categorical_features)])

pipe = Pipeline([('preprocessor', preprocessor), ('classifier', XGBClassifier(random_state=42, n_jobs=1, eval_metric='logloss'))])

param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [4, 6, 8],
    'classifier__learning_rate': [0.05, 0.1],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe, param_grid, scoring='roc_auc', cv=cv, n_jobs=1, verbose=1)
grid.fit(X_train, y_train)

print(f"\nGrid search done at {time.time()-t0:.0f}s")
print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

test_summary = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_proba)
}
print("\nTuned XGBoost -- Test set:")
for m, v in test_summary.items():
    print(f"  {m:10s}: {v:.4f}")

cm = confusion_matrix(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_proba)

joblib.dump(best_model, '../models/best_model_xgboost_tuned.joblib')
with open('../outputs/best_model_params.json', 'w') as f:
    json.dump({'best_params': grid.best_params_, 'best_cv_roc_auc': grid.best_score_,
                'test_metrics': test_summary}, f, indent=2)
np.save('../outputs/confusion_matrix.npy', cm)
np.save('../outputs/roc_curve.npy', np.array([fpr, tpr], dtype=object), allow_pickle=True)

print("\nConfusion matrix:\n", cm)
print(f"\nTotal time: {time.time()-t0:.0f}s")

Fitting 5 folds for each of 12 candidates, totalling 60 fits

Grid search done at 61s
Best params: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 8, 'classifier__n_estimators': 200}
Best CV ROC-AUC: 0.9777212247213487

Tuned XGBoost -- Test set:
  accuracy  : 0.9291
  precision : 0.8537
  recall    : 0.7804
  f1        : 0.8154
  roc_auc   : 0.9772

Confusion matrix:
 [[18542   644]
 [ 1057  3757]]

Total time: 62s
